In [ ]:
pip install tabulate

In [ ]:
import os
import json
import time
import ntpath
from requests import get, post

# ============================================================
# Analyse d'une facture avec Azure Document Intelligence (invoice)
# - Appel POST /analyze (prebuilt-invoice)
# - Polling GET sur operation-location jusqu'à succeeded/failed
# - Cache sur disque: si .invoice.json existe déjà => on le relit
# ============================================================

def analyzeInvoice(filename, output_dir):
    invoiceResultsFilename = os.path.join(
        output_dir,
        ntpath.basename(filename) + ".invoice.json"
    )

    # do not run analyze if .invoice.json file is present on disk
    if os.path.isfile(invoiceResultsFilename):
        with open(invoiceResultsFilename, "r", encoding="utf-8") as json_file:
            return json.load(json_file)

    # Endpoint URL (dans le PDF: endpoint APIM /project-csrd) 【1-9cec35】
    endpoint = os.getenv("DI_ENDPOINT", "https://frc-prd-aai-apim.azure-api.net/project-csrd")
    apim_key = os.getenv("DI_APIM_KEY", "")  # ⚠️ mettre la clé ici via variable d'environnement

    # Dans le PDF: /formrecognizer/documentModels/prebuilt-invoice:analyze 【1-9cec35】
    post_url = endpoint + "/formrecognizer/documentModels/prebuilt-invoice:analyze"

    # Request headers (dans le PDF: Content-Type octet-stream + Ocp-Apim-Subscription-Key) 【1-9cec35】
    headers = {
        "Content-Type": "application/octet-stream",
        "Ocp-Apim-Subscription-Key": apim_key,
    }

    # Params (dans le PDF: includeTextDetails True, api-version 2023-07-31) 【1-9cec35】
    params = {
        "includeTextDetails": True,
        "api-version": "2023-07-31",
    }

    with open(filename, "rb") as f:
        data_bytes = f.read()

    # POST analyze
    try:
        resp = post(url=post_url, data=data_bytes, headers=headers, params=params)
        if resp.status_code != 202:
            print("POST analyze failed:\n%s" % resp.text)
            return None

        print("POST analyze succeeded: %s" % resp.headers["operation-location"])
        raw_get_url = resp.headers["operation-location"]

        # Dans le PDF: get_url = endpoint + raw_get_url[raw_get_url.find(\"/formrecognizer\"):] 【1-9cec35】
        get_url = endpoint + raw_get_url[raw_get_url.find("/formrecognizer"):]

    except Exception as e:
        print("POST analyze failed:\n%s" % str(e))
        return None

    # Polling GET (dans le PDF: n_tries=50 wait_sec=6) 【1-9cec35】
    n_tries = 50
    n_try = 0
    wait_sec = 6

    while n_try < n_tries:
        try:
            resp = get(url=get_url, headers={"Ocp-Apim-Subscription-Key": apim_key})
            resp_json = json.loads(resp.text)

            if resp.status_code != 200:
                print("GET Invoice results failed:\n%s" % resp_json)
                return None

            status = resp_json["status"]

            if status == "succeeded":
                print("Invoice analysis succeeded.")
                with open(invoiceResultsFilename, "w", encoding="utf-8") as outfile:
                    json.dump(resp_json, outfile, indent=4, ensure_ascii=False)
                return resp_json

            if status == "failed":
                print("Analysis failed:\n%s" % resp_json)
                return None

            # Analysis still running. Wait and retry.
            time.sleep(wait_sec)
            n_try += 1

        except Exception as e:
            msg = "GET analyze results failed:\n%s" % str(e)
            print(msg)
            return None

    print("Invoice analyze is not complete after {0} seconds:\n{1}".format(n_try * wait_sec, resp_json))
    return None


# ============================================================
# Main: parcourt un fichier ou un dossier ./invoices/
# et analyse tous les documents supportés en sortant des JSON
# ============================================================

def main():
    path_to_sample_documents = os.path.abspath(
        os.path.join(os.path.abspath(""), "./invoices/")
    )
    output_dir = os.path.join(path_to_sample_documents, "output")

    # Create output directory if it doesn't exist (dans le PDF) 【1-9cec35】
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # List of invoice to analyze
    invoiceFiles = []

    if os.path.isfile(path_to_sample_documents):
        # Single invoice
        invoiceFiles.append(path_to_sample_documents)
    else:
        # Folder of invoices
        supportedExt = [".pdf", ".jpg", ".jpeg", ".tif", ".tiff", ".png", ".bmp"]
        invoiceDirectory = path_to_sample_documents

        for root, directories, filenames in os.walk(invoiceDirectory):
            for invoiceFilename in filenames:
                ext = os.path.splitext(invoiceFilename)[-1].lower()
                if ext in supportedExt:
                    fullname = os.path.join(root, invoiceFilename)
                    invoiceFiles.append(fullname)

    counter = 0
    for invoiceFullFilename in invoiceFiles:
        print(invoiceFullFilename)
        counter += 1

        invoiceFilename = ntpath.basename(invoiceFullFilename)
        print("\n---\nProcessing {0}/{1} : {2}\n".format(counter, len(invoiceFiles), invoiceFullFilename))

        resp_json = analyzeInvoice(invoiceFullFilename, output_dir)
        # (le PDF montre l'appel, sans traitement additionnel derrière) 【1-9cec35】


if __name__ == "__main__":
    main()